In [29]:
# 02_data_quality_review_and_cleaning.ipynb
# Insurance Business Analytics Case Study
# Targeted cleaning for KPI analysis and dashboard preparation

import os
import numpy as np
import pandas as pd


In [30]:
# Paths and raw data 
BASE_DIR = os.getcwd()
RAW_DIR = os.path.join(BASE_DIR, "data_raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "data_processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

customers = pd.read_csv(os.path.join(RAW_DIR, "customers.csv"))
vehicles = pd.read_csv(os.path.join(RAW_DIR, "vehicles.csv"))
policies = pd.read_csv(os.path.join(RAW_DIR, "policies.csv"))
driving = pd.read_csv(os.path.join(RAW_DIR, "driving_behavior.csv"))
tickets = pd.read_csv(os.path.join(RAW_DIR, "support_tickets.csv"))
claims = pd.read_csv(os.path.join(RAW_DIR, "claims.csv"))
financial = pd.read_csv(os.path.join(RAW_DIR, "financial_summary.csv"))

In [31]:
# Quick inspection: where is the mess

print("Raw table shapes:")
for name, df in {
    "customers": customers,
    "vehicles": vehicles,
    "policies": policies,
    "driving": driving,
    "tickets": tickets,
    "claims": claims,
    "financial": financial
}.items():
    print(f"{name}: {df.shape}")

print("\nNull hotspots:")
for name, df in {
    "customers": customers,
    "vehicles": vehicles,
    "policies": policies,
    "driving": driving,
    "tickets": tickets,
    "claims": claims,
    "financial": financial
}.items():
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0].sort_values(ascending=False)
    print(f"\n{name}")
    print(nulls.head(10) if len(nulls) > 0 else "No nulls")

print("\nCategory mess preview:")
print("\npolicy_status")
print(policies["policy_status"].astype(str).value_counts(dropna=False).head(15))

print("\nissue_type")
print(tickets["issue_type"].astype(str).value_counts(dropna=False).head(15))

print("\nsupport_channel")
print(tickets["support_channel"].astype(str).value_counts(dropna=False).head(15))

print("\nclaim_status")
print(claims["claim_status"].astype(str).value_counts(dropna=False).head(15))

Raw table shapes:
customers: (1100, 18)
vehicles: (1200, 14)
policies: (1260, 21)
driving: (12896, 17)
tickets: (2418, 15)
claims: (260, 13)
financial: (1260, 10)

Null hotspots:

customers
email          68
income_band    51
dtype: int64

vehicles
garage_type    83
dtype: int64

policies
cancellation_reason        920
telematics_discount_pct     76
bundled_discount_pct        61
dtype: int64

driving
mileage_logged    667
dtype: int64

tickets
ticket_close_date          718
csat_score                 701
refund_or_credit_amount     73
dtype: int64

claims
approved_claim_amount    85
dtype: int64

financial
No nulls

Category mess preview:

policy_status
active        626
renewed       233
cancelled     203
lapsed        136
ACTIVE         10
 active         9
Active          9
 lapsed         5
active          4
lapsed          4
cancelled       4
RENEWED         3
 renewed        3
Cancelled       2
 cancelled      2
Name: policy_status, dtype: int64

issue_type
mileage_sync_issue   

In [32]:
# Minimal helper functions

def clean_text(x):
    if pd.isna(x):
        return x
    return str(x).strip()

def clean_label(x):
    if pd.isna(x):
        return x
    return str(x).strip().lower().replace(" ", "_")

def clean_phone(x):
    if pd.isna(x):
        return x
    digits = "".join(ch for ch in str(x) if ch.isdigit())
    if len(digits) == 11 and digits.startswith("1"):
        digits = digits[1:]
    return digits if digits else np.nan

In [33]:
# Customers
customers = customers.copy()

customers["customer_id"] = customers["customer_id"].apply(clean_text)
customers["state"] = customers["state"].apply(clean_text).str.title()
customers["city"] = customers["city"].apply(clean_text).str.title()
customers["income_band"] = customers["income_band"].apply(clean_label)
customers["credit_tier"] = customers["credit_tier"].apply(clean_label)
customers["marital_status"] = customers["marital_status"].apply(clean_label)
customers["customer_segment"] = customers["customer_segment"].apply(clean_label)
customers["phone_number"] = customers["phone_number"].apply(clean_phone)

customers["date_of_birth"] = pd.to_datetime(customers["date_of_birth"], errors="coerce")
customers["customer_join_date"] = pd.to_datetime(customers["customer_join_date"], errors="coerce")
customers["age"] = pd.to_numeric(customers["age"], errors="coerce")
customers["tenure_months"] = pd.to_numeric(customers["tenure_months"], errors="coerce")
customers["household_size"] = pd.to_numeric(customers["household_size"], errors="coerce")

# keep first customer_id if duplicates exist
customers = customers.drop_duplicates(subset="customer_id", keep="first")

# only light validity checks
customers.loc[(customers["age"] < 18) | (customers["age"] > 90), "age"] = np.nan
customers.loc[(customers["tenure_months"] < 0), "tenure_months"] = np.nan


In [34]:
# Vehicles
vehicles = vehicles.copy()

vehicles["vehicle_id"] = vehicles["vehicle_id"].apply(clean_text)
vehicles["customer_id"] = vehicles["customer_id"].apply(clean_text)
vehicles["vehicle_make"] = vehicles["vehicle_make"].apply(clean_text).str.title()
vehicles["vehicle_model"] = vehicles["vehicle_model"].apply(clean_text).str.title()
vehicles["vehicle_type"] = vehicles["vehicle_type"].apply(clean_label)
vehicles["fuel_type"] = vehicles["fuel_type"].apply(clean_label)
vehicles["transmission_type"] = vehicles["transmission_type"].apply(clean_label)
vehicles["primary_use"] = vehicles["primary_use"].apply(clean_label)
vehicles["ownership_status"] = vehicles["ownership_status"].apply(clean_label)
vehicles["garage_type"] = vehicles["garage_type"].apply(clean_label)

vehicles["vehicle_year"] = pd.to_numeric(vehicles["vehicle_year"], errors="coerce")
vehicles["vehicle_value_estimate"] = pd.to_numeric(vehicles["vehicle_value_estimate"], errors="coerce")
vehicles["annual_mileage"] = pd.to_numeric(vehicles["annual_mileage"], errors="coerce")
vehicles["anti_theft_device_flag"] = pd.to_numeric(vehicles["anti_theft_device_flag"], errors="coerce")

vehicles = vehicles.drop_duplicates(subset="vehicle_id", keep="first")

# leave some missing / ugly values alone if they do not break the analysis
vehicles.loc[(vehicles["vehicle_value_estimate"] < 1000) | (vehicles["vehicle_value_estimate"] > 250000), "vehicle_value_estimate"] = np.nan



In [35]:
# Policies
policies = policies.copy()

policies["policy_id"] = policies["policy_id"].apply(clean_text)
policies["customer_id"] = policies["customer_id"].apply(clean_text)
policies["vehicle_id"] = policies["vehicle_id"].apply(clean_text)

for col in ["policy_type", "coverage_level", "payment_frequency", "policy_status", "cancellation_reason"]:
    policies[col] = policies[col].apply(clean_label)

policies["policy_start_date"] = pd.to_datetime(policies["policy_start_date"], errors="coerce")
policies["policy_end_date"] = pd.to_datetime(policies["policy_end_date"], errors="coerce")

numeric_policy_cols = [
    "deductible", "risk_score", "base_premium", "telematics_discount_pct",
    "loyalty_discount_pct", "bundled_discount_pct", "final_monthly_premium",
    "autopay_flag", "renewal_flag", "churn_flag", "payment_timeliness_score"
]
for col in numeric_policy_cols:
    policies[col] = pd.to_numeric(policies[col], errors="coerce")

policies = policies.drop_duplicates(subset="policy_id", keep="first")

# keep outliers visible, but neutralize the most obviously broken ranges
policies.loc[(policies["final_monthly_premium"] < 10) | (policies["final_monthly_premium"] > 5000), "final_monthly_premium"] = np.nan
policies.loc[(policies["risk_score"] < 0) | (policies["risk_score"] > 5), "risk_score"] = np.nan

In [36]:
# Driving behavior
driving = driving.copy()

for col in ["driving_record_id", "policy_id", "customer_id", "vehicle_id"]:
    driving[col] = driving[col].apply(clean_text)

driving["observation_month"] = pd.to_datetime(driving["observation_month"], errors="coerce")

numeric_driving_cols = [
    "driving_score", "avg_trip_distance", "trips_per_week",
    "hard_brakes_count", "rapid_acceleration_count",
    "night_driving_pct", "weekend_driving_pct",
    "distracted_driving_flag", "speeding_events_count",
    "urban_driving_pct", "highway_driving_pct", "mileage_logged"
]
for col in numeric_driving_cols:
    driving[col] = pd.to_numeric(driving[col], errors="coerce")

driving = driving.drop_duplicates(subset="driving_record_id", keep="first")

# do not clean every weird thing here; only fix values that would destroy summary logic
driving.loc[(driving["driving_score"] < 0) | (driving["driving_score"] > 100), "driving_score"] = np.nan
driving.loc[(driving["mileage_logged"] < 0), "mileage_logged"] = np.nan

for col in ["night_driving_pct", "weekend_driving_pct", "urban_driving_pct", "highway_driving_pct"]:
    driving.loc[(driving[col] < 0) | (driving[col] > 1), col] = np.nan

In [37]:
# Support tickets
tickets = tickets.copy()

tickets["ticket_id"] = tickets["ticket_id"].apply(clean_text)
tickets["customer_id"] = tickets["customer_id"].apply(clean_text)
tickets["policy_id"] = tickets["policy_id"].apply(clean_text)

for col in ["issue_type", "issue_severity", "support_channel", "resolution_status"]:
    tickets[col] = tickets[col].apply(clean_label)

tickets["ticket_open_date"] = pd.to_datetime(tickets["ticket_open_date"], errors="coerce")
tickets["ticket_close_date"] = pd.to_datetime(tickets["ticket_close_date"], errors="coerce")

numeric_ticket_cols = [
    "resolution_time_hours", "escalation_flag", "repeat_ticket_flag",
    "csat_score", "refund_or_credit_given_flag", "refund_or_credit_amount"
]
for col in numeric_ticket_cols:
    tickets[col] = pd.to_numeric(tickets[col], errors="coerce")

tickets = tickets.drop_duplicates(subset="ticket_id", keep="first")

# clean only what matters for ops KPIs
tickets.loc[(tickets["resolution_time_hours"] < 0), "resolution_time_hours"] = np.nan
tickets.loc[(tickets["csat_score"] < 1) | (tickets["csat_score"] > 5), "csat_score"] = np.nan
tickets.loc[(tickets["refund_or_credit_amount"] < 0), "refund_or_credit_amount"] = np.nan

In [38]:
# Claims
claims = claims.copy()

claims["claim_id"] = claims["claim_id"].apply(clean_text)
claims["customer_id"] = claims["customer_id"].apply(clean_text)
claims["policy_id"] = claims["policy_id"].apply(clean_text)
claims["vehicle_id"] = claims["vehicle_id"].apply(clean_text)

for col in ["claim_type", "claim_severity", "claim_status", "fault_assignment"]:
    claims[col] = claims[col].apply(clean_label)

claims["claim_date"] = pd.to_datetime(claims["claim_date"], errors="coerce")

for col in ["estimated_claim_cost", "approved_claim_amount", "fraud_risk_flag", "weather_related_flag"]:
    claims[col] = pd.to_numeric(claims[col], errors="coerce")

claims = claims.drop_duplicates(subset="claim_id", keep="first")

claims.loc[(claims["estimated_claim_cost"] < 0), "estimated_claim_cost"] = np.nan
claims.loc[(claims["approved_claim_amount"] < 0), "approved_claim_amount"] = np.nan

In [39]:
# Financial summary
financial = financial.copy()

financial["customer_id"] = financial["customer_id"].apply(clean_text)
financial["policy_id"] = financial["policy_id"].apply(clean_text)
financial["vehicle_id"] = financial["vehicle_id"].apply(clean_text)
financial["profitability_segment"] = financial["profitability_segment"].apply(clean_label)

for col in [
    "total_premium_paid", "total_discount_amount", "total_support_cost",
    "total_claim_cost", "estimated_customer_ltv", "estimated_net_value"
]:
    financial[col] = pd.to_numeric(financial[col], errors="coerce")

financial = financial.drop_duplicates(subset="policy_id", keep="first")

In [40]:
# Relationship checks: remove records that break the model

print("\nRelationship issues before filtering:")
print("vehicles with missing customer:", (~vehicles["customer_id"].isin(customers["customer_id"])).sum())
print("policies with missing customer:", (~policies["customer_id"].isin(customers["customer_id"])).sum())
print("policies with missing vehicle:", (~policies["vehicle_id"].isin(vehicles["vehicle_id"])).sum())
print("tickets with missing policy:", (~tickets["policy_id"].isin(policies["policy_id"])).sum())
print("claims with missing policy:", (~claims["policy_id"].isin(policies["policy_id"])).sum())

vehicles = vehicles[vehicles["customer_id"].isin(customers["customer_id"])].copy()

policies = policies[
    policies["customer_id"].isin(customers["customer_id"]) &
    policies["vehicle_id"].isin(vehicles["vehicle_id"])
].copy()

tickets = tickets[
    tickets["customer_id"].isin(customers["customer_id"]) &
    tickets["policy_id"].isin(policies["policy_id"])
].copy()

claims = claims[
    claims["customer_id"].isin(customers["customer_id"]) &
    claims["policy_id"].isin(policies["policy_id"]) &
    claims["vehicle_id"].isin(vehicles["vehicle_id"])
].copy()

driving = driving[
    driving["customer_id"].isin(customers["customer_id"]) &
    driving["policy_id"].isin(policies["policy_id"]) &
    driving["vehicle_id"].isin(vehicles["vehicle_id"])
].copy()

financial = financial[
    financial["customer_id"].isin(customers["customer_id"]) &
    financial["policy_id"].isin(policies["policy_id"]) &
    financial["vehicle_id"].isin(vehicles["vehicle_id"])
].copy()


Relationship issues before filtering:
vehicles with missing customer: 0
policies with missing customer: 0
policies with missing vehicle: 0
tickets with missing policy: 0
claims with missing policy: 0


In [41]:
# Derived fields for KPI sections

# Policy-level bands for churn / profitability / dashboard filters
policies["premium_band"] = pd.cut(
    policies["final_monthly_premium"],
    bins=[0, 80, 140, 220, 999999],
    labels=["low", "mid", "high", "very_high"]
).astype("object")

policies["total_discount_pct"] = (
    policies[["telematics_discount_pct", "loyalty_discount_pct", "bundled_discount_pct"]]
    .fillna(0)
    .sum(axis=1)
)

policies["discount_band"] = pd.cut(
    policies["total_discount_pct"],
    bins=[-1, 0, 5, 10, 20, 100],
    labels=["none", "low", "moderate", "high", "very_high"]
).astype("object")

policies["risk_band"] = pd.cut(
    policies["risk_score"],
    bins=[0, 0.6, 0.9, 1.2, 10],
    labels=["low_risk", "moderate_risk", "high_risk", "very_high_risk"]
).astype("object")

In [42]:
# Ticket-level support burden by policy
ticket_summary = (
    tickets.groupby("policy_id", as_index=False)
    .agg(
        ticket_count=("ticket_id", "count"),
        avg_resolution_time_hours=("resolution_time_hours", "mean"),
        avg_csat=("csat_score", "mean"),
        escalation_rate=("escalation_flag", "mean"),
        refund_rate=("refund_or_credit_given_flag", "mean"),
        total_refund_amount=("refund_or_credit_amount", "sum")
    )
)

ticket_summary["support_burden_band"] = pd.cut(
    ticket_summary["ticket_count"],
    bins=[-1, 0, 2, 5, 999],
    labels=["none", "low", "moderate", "high"]
).astype("object")

# Claim-level summary by policy
claim_summary = (
    claims.groupby("policy_id", as_index=False)
    .agg(
        claim_count=("claim_id", "count"),
        total_estimated_claim_cost=("estimated_claim_cost", "sum"),
        total_approved_claim_amount=("approved_claim_amount", "sum")
    )
)

In [43]:
# Build dashboard-ready analytical tables
# policy_master
# One row per policy
policy_master = (
    policies
    .merge(customers[[
        "customer_id", "age", "state", "city", "income_band",
        "credit_tier", "customer_segment", "tenure_months"
    ]], on="customer_id", how="left")
    .merge(vehicles[[
        "vehicle_id", "vehicle_make", "vehicle_model", "vehicle_year",
        "vehicle_type", "vehicle_value_estimate", "annual_mileage", "primary_use"
    ]], on="vehicle_id", how="left")
    .merge(financial[[
        "policy_id", "total_premium_paid", "total_discount_amount",
        "total_support_cost", "total_claim_cost",
        "estimated_customer_ltv", "estimated_net_value", "profitability_segment"
    ]], on="policy_id", how="left")
    .merge(ticket_summary, on="policy_id", how="left")
    .merge(claim_summary, on="policy_id", how="left")
)

# fill summary gaps for policies with no tickets / no claims
for col in [
    "ticket_count", "avg_resolution_time_hours", "avg_csat", "escalation_rate",
    "refund_rate", "total_refund_amount",
    "claim_count", "total_estimated_claim_cost", "total_approved_claim_amount",
    "total_support_cost", "total_claim_cost"
]:
    if col in policy_master.columns:
        policy_master[col] = policy_master[col].fillna(0)

policy_master["support_burden_band"] = policy_master["support_burden_band"].fillna("none")
policy_master["profitability_segment"] = policy_master["profitability_segment"].fillna("unknown")

In [44]:
# ticket_ops
# One row per ticket for operations dashboarding
ticket_ops = (
    tickets
    .merge(policies[[
        "policy_id", "policy_type", "coverage_level", "policy_status",
        "renewal_flag", "churn_flag", "premium_band", "discount_band", "risk_band"
    ]], on="policy_id", how="left")
    .merge(customers[[
        "customer_id", "customer_segment", "state", "income_band"
    ]], on="customer_id", how="left")
)

# claims_kpi_table
# Optional but useful
claims_kpi_table = (
    claims
    .merge(policies[[
        "policy_id", "policy_type", "coverage_level", "risk_band",
        "final_monthly_premium", "premium_band"
    ]], on="policy_id", how="left")
    .merge(vehicles[[
        "vehicle_id", "vehicle_type", "vehicle_make", "vehicle_model"
    ]], on="vehicle_id", how="left")
)



In [45]:
# What we intentionally did NOT clean too much

print("\nNotes:")
print("- Some non-critical missing values were left in place.")
print("- Some operational weirdness was preserved if it did not break KPI logic.")
print("- Extreme but plausible claim / premium behavior was not fully sanitized.")
print("- The goal was usable business data, not artificially perfect data.")


Notes:
- Some non-critical missing values were left in place.
- Some operational weirdness was preserved if it did not break KPI logic.
- Extreme but plausible claim / premium behavior was not fully sanitized.
- The goal was usable business data, not artificially perfect data.


In [46]:
# 9. Save cleaned and analytical outputs

customers.to_csv(os.path.join(PROCESSED_DIR, "customers_clean.csv"), index=False)
vehicles.to_csv(os.path.join(PROCESSED_DIR, "vehicles_clean.csv"), index=False)
policies.to_csv(os.path.join(PROCESSED_DIR, "policies_clean.csv"), index=False)
driving.to_csv(os.path.join(PROCESSED_DIR, "driving_behavior_clean.csv"), index=False)
tickets.to_csv(os.path.join(PROCESSED_DIR, "support_tickets_clean.csv"), index=False)
claims.to_csv(os.path.join(PROCESSED_DIR, "claims_clean.csv"), index=False)
financial.to_csv(os.path.join(PROCESSED_DIR, "financial_summary_clean.csv"), index=False)

policy_master.to_csv(os.path.join(PROCESSED_DIR, "policy_master.csv"), index=False)
ticket_ops.to_csv(os.path.join(PROCESSED_DIR, "ticket_ops.csv"), index=False)
claims_kpi_table.to_csv(os.path.join(PROCESSED_DIR, "claims_kpi_table.csv"), index=False)

In [47]:
# Final output check

print("\nFinal output shapes:")
print("customers_clean:", customers.shape)
print("vehicles_clean:", vehicles.shape)
print("policies_clean:", policies.shape)
print("driving_clean:", driving.shape)
print("support_tickets_clean:", tickets.shape)
print("claims_clean:", claims.shape)
print("financial_summary_clean:", financial.shape)
print("policy_master:", policy_master.shape)
print("ticket_ops:", ticket_ops.shape)
print("claims_kpi_table:", claims_kpi_table.shape)

print("\nFiles saved to:", PROCESSED_DIR)


Final output shapes:
customers_clean: (1100, 18)
vehicles_clean: (1200, 14)
policies_clean: (1260, 25)
driving_clean: (12896, 17)
support_tickets_clean: (2418, 15)
claims_clean: (260, 13)
financial_summary_clean: (1260, 10)
policy_master: (1260, 56)
ticket_ops: (2418, 26)
claims_kpi_table: (260, 21)

Files saved to: C:\Users\thebe\OneDrive\Desktop\IsuranceDA\data_processed


In [50]:
print ("Only the fields needed for joins, grouping, KPI calculations, and dashboard filters were standardized. Non-critical irregularities were intentionally preserved.")

Only the fields needed for joins, grouping, KPI calculations, and dashboard filters were standardized. Non-critical irregularities were intentionally preserved.
